In [ ]:
# 1) 의존성 설치
!pip install --no-deps protenix biopython ml-collections biotite==1.0.1 rdkit

In [ ]:
# 2) 환경변수 & 체크포인트 경로
import os, sys, torch
os.environ['PROTENIX_DATA_ROOT_DIR'] = '/kaggle/input/protenix-checkpoints'

In [ ]:
# 3) USalign 준비
!cp /kaggle/input/usalign/USalign /kaggle/working/
!chmod +x /kaggle/working/USalign
USALIGN = '/kaggle/working/USalign'

In [ ]:
# 4) 기본 임포트
import pandas as pd
import numpy as np
from tqdm import tqdm
from runner.batch_inference import get_default_runner
from runner.inference import update_inference_configs, InferenceRunner
from protenix.data.infer_data_pipeline import InferenceDataset
from configs.configs_base import configs as configs_base
from configs.configs_data import data_configs
from configs.configs_inference import inference_configs
from protenix.config.config import parse_configs

In [ ]:
# 5) 파서·유틸 정의
import re
from Bio.PDB import PDBParser

def write_target_line(atom_name, atom_serial, residue_name, chain_id, residue_num, x, y, z, occupancy=1.0, b_factor=0.0, atom_type='P'):
    return f'ATOM  {atom_serial:>5d}  {atom_name:<5s} {residue_name:<3s} {residue_num:>3d}    {x:>8.3f}{y:>8.3f}{z:>8.3f}{occupancy:>6.2f}{b_factor:>6.2f}           {atom_type}\n'

def write_xyz_to_pdb(df, out_path, xyz_id=1):
    with open(out_path, 'w') as f:
        for _, r in df.iterrows():
            x,y,z = r[f'x_{xyz_id}'], r[f'y_{xyz_id}'], r[f'z_{xyz_id}']
            f.write(write_target_line("C1'", int(r.resid), r.resname, 'A', int(r.resid), x,y,z))
            
def parse_usalign_for_tm_score(txt):
    m = re.findall(r'TM-score=\s*([\d.]+)', txt)
    return float(m[1])

def parse_usalign_for_transform(txt):
    lines, mat, flag = [], [], False
    for L in txt.splitlines():
        if "The rotation matrix to rotate Structure_1 to Structure_2" in L:
            flag=True
        elif flag and re.match(r'^\d+\s+', L):
            parts = L.split()
            mat.append(list(map(float, parts[1:])))
        elif flag and not L.strip():
            break
    return np.array(mat)

def call_usalign(pred_df, truth_df):
    write_xyz_to_pdb(pred_df, '/tmp/pred.pdb', xyz_id=1)
    write_xyz_to_pdb(truth_df, '/tmp/truth.pdb', xyz_id=1)
    out = os.popen(f"{USALIGN} /tmp/pred.pdb /tmp/truth.pdb -atom \" C1'\" -m -").read()
    tm = parse_usalign_for_tm_score(out)
    return tm

def parse_output_to_df(output, seq, target_id):
    # output: [N_sample, L, 3]
    records = []
    N_sample, L, _ = output.shape
    for i in range(L):
        base = {"ID": target_id, "resname": seq[i], "resid": i+1}
        for k in range(N_sample):
            base[f"x_{k+1}"] = round(output[k,i,0].item(),3)
            base[f"y_{k+1}"] = round(output[k,i,1].item(),3)
            base[f"z_{k+1}"] = round(output[k,i,2].item(),3)
        records.append(base.copy())
    return pd.DataFrame(records)

In [ ]:
# 6) DictDataset 정의 (MSA 미사용)
class DictDataset(InferenceDataset):
    def __init__(self, seq_list, dump_dir, id_list=None, use_msa=False):
        self.dump_dir = dump_dir
        self.use_msa  = use_msa
        if id_list is None:
            self.inputs = [{"sequences":[{"rnaSequence":{"sequence":s,"count":1}}],"name":"query"} for s in seq_list]
        else:
            self.inputs = [{"sequences":[{"rnaSequence":{"sequence":s,"count":1}}],"name":tid}
                           for s,tid in zip(seq_list, id_list)]

In [ ]:
# 7) Runner 세팅
torch.manual_seed(0)
np.random.seed(0)
torch.cuda.manual_seed_all(0)

configs_base["use_deepspeed_evo_attention"]   = False
# 디퓨전모델 파라미터 세팅
configs_base["model"]["N_cycle"]             = 10
configs_base["sample_diffusion"]["N_sample"] = 5
configs_base["sample_diffusion"]["N_step"]   = 150
inference_configs['load_checkpoint_path']    = '/kaggle/input/protenix-checkpoints/model_v0.2.0.pt'

configs = {**configs_base, **{"data": data_configs}, **inference_configs}
configs = parse_configs(configs=configs, fill_required_with_null=True)
runner = InferenceRunner(configs)

In [ ]:
# 8) 제출 루프
test_df = pd.read_csv('/kaggle/input/stanford-rna-3d-folding/test_sequences.csv')

dataset = DictDataset(
    seq_list=test_df.sequence.tolist(),
    dump_dir='output_submit',
    id_list=test_df.target_id.tolist(),
    use_msa=False
)

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    seq, tid = row.sequence, row.target_id

    data, _, err = dataset[i]
    if err:
        dummy = np.zeros((5, len(seq), 3))
        df = parse_output_to_df(dummy, seq, tid)
    else:
        coords = runner.predict(data)['coordinate'][
            :, data['input_feature_dict']['atom_to_tokatom_idx']==12
        ]  # shape: [5, n_residues, 3]

        df = parse_output_to_df(coords, seq, tid)

    df['ID'] = df.apply(lambda r: f"{r.ID}_{int(r.resid)}", axis=1)
    df.to_csv('submission.csv', index=False, mode='a', header=(i==0))


In [ ]:
# 9) 제출 파일 확인
print("Submission created:", len(test_df), "targets")
display(pd.read_csv('submission.csv'))